# FX Portfolio Optimization with FXMacroData

This tutorial uses the official `fxmacrodata` Python client to fetch daily FX reference-rate history, converts the price series into aligned returns, and then optimizes a long-only FX portfolio with Riskfolio-Lib.

The client reads `FXMACRODATA_API_KEY` or `FXMD_API_KEY` from the environment. The chosen pairs require a key for full history; never paste a key into this notebook.

In [ ]:
# pip install riskfolio-lib fxmacrodata pandas

import os

import pandas as pd
import riskfolio as rp
from fxmacrodata import FXMacroDataClient


In [ ]:
def fetch_fx_prices(
    client: FXMacroDataClient,
    pairs: dict[str, tuple[str, str]],
    start_date: str,
    end_date: str,
) -> pd.DataFrame:
    columns = []
    for asset, (base, quote) in pairs.items():
        rows = client.forex(base, quote, start_date=start_date, end_date=end_date)
        frame = pd.DataFrame(rows)
        if not {"date", "val"}.issubset(frame.columns):
            raise RuntimeError(f"{asset} response did not contain date and val fields")
        series = pd.Series(
            pd.to_numeric(frame["val"], errors="coerce").to_numpy(),
            index=pd.to_datetime(frame["date"], utc=True),
            name=asset,
        )
        columns.append(series.groupby(level=0).last())

    prices = pd.concat(columns, axis=1).sort_index().ffill().dropna()
    if prices.shape[1] < 2 or len(prices) < 2:
        raise RuntimeError("at least two aligned FX price series are required")
    return prices


In [ ]:
api_key = os.environ.get("FXMACRODATA_API_KEY") or os.environ.get("FXMD_API_KEY")
if not api_key:
    raise RuntimeError("Set FXMACRODATA_API_KEY or FXMD_API_KEY before requesting protected FX history")

client = FXMacroDataClient(api_key=api_key, auth_mode="query")
pairs = {
    "EURUSD": ("EUR", "USD"),
    "GBPUSD": ("GBP", "USD"),
    "AUDUSD": ("AUD", "USD"),
    "NZDUSD": ("NZD", "USD"),
}
prices = fetch_fx_prices(client, pairs, start_date="2023-01-01", end_date="2025-12-31")
returns = prices.pct_change().dropna()
returns.head()


In [ ]:
portfolio = rp.Portfolio(returns=returns)
portfolio.assets_stats(method_mu="hist", method_cov="hist")

weights = portfolio.optimization(
    model="Classic",
    rm="MV",
    obj="Sharpe",
    rf=0,
    l=0,
    hist=True,
)
weights.T


## Notes

- FXMacroData reference rates are daily official-source values, not executable quotes.
- This example optimizes historical returns; it does not place orders or make portfolio recommendations.
- For release-aware research, join FXMacroData announcement rows by their `announcement_datetime` and ensure no value is used before its release time.